In [0]:
dbutils.widgets.removeAll()

from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("catalogo", "retail_dev")
dbutils.widgets.text("esquema_origen", "bronze")
dbutils.widgets.text("esquema_destino", "silver")

catalogo = dbutils.widgets.get("catalogo")
esquema_origen = dbutils.widgets.get("esquema_origen")
esquema_destino = dbutils.widgets.get("esquema_destino")

tabla_inventario = (
    f"{catalogo}.{esquema_origen}.inventario"
)

tabla_productos = (
    f"{catalogo}.{esquema_origen}.productos"
)

tabla_destino = (
    f"{catalogo}.{esquema_destino}.inventario_limpio"
)

print(f"Tabla inventario: {tabla_inventario}")
print(f"Tabla productos: {tabla_productos}")
print(f"Tabla destino: {tabla_destino}")

In [0]:
df_inventario_raw = spark.table(tabla_inventario)
df_productos_raw = spark.table(tabla_productos)

print(
    f"Registros de inventario en Bronze: "
    f"{df_inventario_raw.count()}"
)

print(
    f"Registros de productos en Bronze: "
    f"{df_productos_raw.count()}"
)

In [0]:
print("Columnas de inventario:")
print(df_inventario_raw.columns)

print("\nColumnas de productos:")
print(df_productos_raw.columns)

df_inventario_raw.printSchema()
df_productos_raw.printSchema()

In [0]:
df_productos_limpios = (
    df_productos_raw

    .withColumn(
        "producto_id_limpio",
        F.expr(
            """
            try_cast(
                try_cast(trim(producto_id) AS DOUBLE)
                AS INT
            )
            """
        )
    )

    .withColumn(
        "sku_limpio",
        F.when(
            F.col("sku").isNull()
            | (F.trim(F.col("sku")) == ""),
            None
        ).otherwise(
            F.upper(F.trim(F.col("sku")))
        )
    )

    .withColumn(
        "producto_limpio",
        F.when(
            F.col("producto").isNull()
            | (F.trim(F.col("producto")) == ""),
            "Producto sin nombre"
        ).otherwise(
            F.initcap(F.trim(F.col("producto")))
        )
    )

    .withColumn(
        "categoria_limpia",
        F.when(
            F.col("categoria").isNull()
            | (F.trim(F.col("categoria")) == ""),
            "Sin categoría"
        ).otherwise(
            F.initcap(F.trim(F.col("categoria")))
        )
    )

    .withColumn(
        "marca_limpia",
        F.when(
            F.col("marca").isNull()
            | (F.trim(F.col("marca")) == ""),
            "Sin marca"
        ).otherwise(
            F.initcap(F.trim(F.col("marca")))
        )
    )

    .withColumn(
        "costo_numero",
        F.expr(
            """
            try_cast(
                regexp_replace(trim(costo), ',', '')
                AS DOUBLE
            )
            """
        )
    )

    .withColumn(
        "costo_limpio",
        F.when(
            F.col("costo_numero") > 0,
            F.col("costo_numero")
        ).otherwise(
            F.lit(0.0)
        )
    )

    .filter(
        F.col("producto_id_limpio").isNotNull()
    )

    .withColumn(
        "_rn",
        F.row_number().over(
            Window
            .partitionBy("producto_id_limpio")
            .orderBy(
                F.col("ingestion_date").desc_nulls_last()
            )
        )
    )

    .filter(F.col("_rn") == 1)

    .select(
        F.col("producto_id_limpio").alias("producto_id"),
        F.col("sku_limpio").alias("sku"),
        F.col("producto_limpio").alias("producto"),
        F.col("categoria_limpia").alias("categoria"),
        F.col("marca_limpia").alias("marca"),
        F.col("costo_limpio").alias("costo")
    )
)

print(
    f"Productos limpios: {df_productos_limpios.count()}"
)

display(df_productos_limpios.limit(20))

In [0]:
df_inventario_limpio_base = (
    df_inventario_raw

    .withColumn(
        "inventario_id_limpio",
        F.expr(
            """
            try_cast(
                try_cast(trim(inventario_id) AS DOUBLE)
                AS INT
            )
            """
        )
    )

    .withColumn(
        "producto_id_limpio",
        F.expr(
            """
            try_cast(
                try_cast(trim(producto_id) AS DOUBLE)
                AS INT
            )
            """
        )
    )

    .withColumn(
        "almacen_limpio",
        F.when(
            F.col("almacen").isNull()
            | (F.trim(F.col("almacen")) == ""),
            "No especificado"
        ).otherwise(
            F.initcap(F.trim(F.col("almacen")))
        )
    )

    .withColumn(
        "stock_actual_numero",
        F.expr(
            """
            try_cast(
                try_cast(trim(stock_actual) AS DOUBLE)
                AS INT
            )
            """
        )
    )

    .withColumn(
        "stock_minimo_numero",
        F.expr(
            """
            try_cast(
                try_cast(trim(stock_minimo) AS DOUBLE)
                AS INT
            )
            """
        )
    )

    .withColumn(
        "stock_actual_limpio",
        F.when(
            F.col("stock_actual_numero").isNull()
            | (F.col("stock_actual_numero") < 0),
            0
        ).otherwise(
            F.col("stock_actual_numero")
        )
    )

    .withColumn(
        "stock_minimo_limpio",
        F.when(
            F.col("stock_minimo_numero").isNull()
            | (F.col("stock_minimo_numero") < 0),
            0
        ).otherwise(
            F.col("stock_minimo_numero")
        )
    )

    .withColumn(
        "fecha_actualizacion_limpia",
        F.expr(
            "try_cast(trim(fecha_actualizacion) AS DATE)"
        )
    )

    .filter(
        F.col("inventario_id_limpio").isNotNull()
        & F.col("producto_id_limpio").isNotNull()
    )

    .withColumn(
        "_rn",
        F.row_number().over(
            Window
            .partitionBy("inventario_id_limpio")
            .orderBy(
                F.col("ingestion_date").desc_nulls_last()
            )
        )
    )

    .filter(F.col("_rn") == 1)

    .select(
        F.col("inventario_id_limpio").alias("inventario_id"),
        F.col("producto_id_limpio").alias("producto_id"),
        F.col("almacen_limpio").alias("almacen"),
        F.col("stock_actual_limpio").alias("stock_actual"),
        F.col("stock_minimo_limpio").alias("stock_minimo"),
        F.col("fecha_actualizacion_limpia").alias(
            "fecha_actualizacion"
        )
    )
)

print(
    f"Inventario limpio: "
    f"{df_inventario_limpio_base.count()}"
)

display(df_inventario_limpio_base.limit(20))

In [0]:
df_inventario_final = (
    df_inventario_limpio_base.alias("i")

    .join(
        df_productos_limpios.alias("p"),
        F.col("i.producto_id")
        == F.col("p.producto_id"),
        "inner"
    )

    .withColumn(
        "estatus_inventario",
        F.when(
            F.col("i.stock_actual") == 0,
            "SIN STOCK"
        )
        .when(
            F.col("i.stock_actual")
            <= F.col("i.stock_minimo"),
            "BAJO STOCK"
        )
        .otherwise(
            "STOCK NORMAL"
        )
    )

    .withColumn(
        "valor_inventario",
        F.round(
            F.col("i.stock_actual")
            * F.col("p.costo"),
            2
        )
    )

    .select(
        F.col("i.inventario_id")
        .cast("int")
        .alias("inventario_id"),

        F.col("p.producto_id")
        .cast("int")
        .alias("producto_id"),

        F.col("p.sku").alias("sku"),
        F.col("p.producto").alias("producto"),
        F.col("p.categoria").alias("categoria"),
        F.col("p.marca").alias("marca"),
        F.col("i.almacen").alias("almacen"),

        F.col("i.stock_actual")
        .cast("int")
        .alias("stock_actual"),

        F.col("i.stock_minimo")
        .cast("int")
        .alias("stock_minimo"),

        F.col("i.fecha_actualizacion")
        .alias("fecha_actualizacion"),

        F.col("estatus_inventario"),

        F.col("valor_inventario")
        .cast("double")
        .alias("valor_inventario")
    )
)

print(
    f"Registros después del JOIN: "
    f"{df_inventario_final.count()}"
)

display(df_inventario_final.limit(20))

In [0]:
df_inventario_final.printSchema()

In [0]:
validacion_previa = df_inventario_final.select(
    F.count("*").alias("registros"),

    F.sum(
        F.when(
            F.col("inventario_id").isNull(),
            1
        ).otherwise(0)
    ).alias("inventario_id_nulo"),

    F.sum(
        F.when(
            F.col("producto_id").isNull(),
            1
        ).otherwise(0)
    ).alias("producto_id_nulo"),

    F.sum(
        F.when(
            F.col("stock_actual") < 0,
            1
        ).otherwise(0)
    ).alias("stock_actual_negativo"),

    F.sum(
        F.when(
            F.col("stock_minimo") < 0,
            1
        ).otherwise(0)
    ).alias("stock_minimo_negativo"),

    F.sum(
        F.when(
            F.col("valor_inventario") < 0,
            1
        ).otherwise(0)
    ).alias("valor_inventario_negativo"),

    F.sum(
        F.when(
            F.col("estatus_inventario").isNull(),
            1
        ).otherwise(0)
    ).alias("estatus_nulo")
)

display(validacion_previa)

In [0]:
(
    df_inventario_final.write
    .mode("overwrite")
    .insertInto(tabla_destino)
)

print(
    f"Tabla cargada correctamente: {tabla_destino}"
)

In [0]:
df_silver_inventario = spark.table(tabla_destino)

cantidad_dataframe = df_inventario_final.count()
cantidad_silver = df_silver_inventario.count()

print(
    f"Registros del DataFrame final: "
    f"{cantidad_dataframe}"
)

print(
    f"Registros cargados en Silver: "
    f"{cantidad_silver}"
)

if cantidad_dataframe != cantidad_silver:
    raise ValueError(
        "La cantidad del DataFrame no coincide con Silver."
    )

print(
    "Carga de inventario_limpio "
    "completada correctamente."
)

display(df_silver_inventario.limit(20))

In [0]:
validacion_final = df_silver_inventario.select(
    F.count("*").alias("registros"),

    F.sum(
        F.when(
            F.col("inventario_id").isNull(),
            1
        ).otherwise(0)
    ).alias("inventario_id_nulo"),

    F.sum(
        F.when(
            F.col("producto_id").isNull(),
            1
        ).otherwise(0)
    ).alias("producto_id_nulo"),

    F.sum(
        F.when(
            F.col("stock_actual") < 0,
            1
        ).otherwise(0)
    ).alias("stock_actual_negativo"),

    F.sum(
        F.when(
            F.col("stock_minimo") < 0,
            1
        ).otherwise(0)
    ).alias("stock_minimo_negativo"),

    F.sum(
        F.when(
            F.col("valor_inventario") < 0,
            1
        ).otherwise(0)
    ).alias("valor_inventario_negativo"),

    F.sum(
        F.when(
            F.col("estatus_inventario").isNull(),
            1
        ).otherwise(0)
    ).alias("estatus_nulo")
)

display(validacion_final)

In [0]:
display(
    df_silver_inventario

    .groupBy("estatus_inventario")

    .agg(
        F.count("*").alias("registros"),

        F.sum("stock_actual").alias(
            "unidades_totales"
        ),

        F.round(
            F.sum("valor_inventario"),
            2
        ).alias(
            "valor_total_inventario"
        )
    )

    .orderBy("estatus_inventario")
)